# **Ovarian Reserve - Inference workflow**
___

This notebook runs the **3D instance segmentation inference** workflow in **BiaPy** for ovarian oocytes, using the ovarian reserve inference template in this repository and the Ovarian Reserve training workflow as context.

It supports three model-loading modes:
- **Official pretrained checkpoint** recommended for this workflow.
- **Any compatible custom `.pth` checkpoint**.
- **A BioImage Model Zoo model ID** through BiaPy.

Recommended pretrained weights:
https://github.com/BiaPyX/BiaPy/blob/8fd93d3167d061896edf940c7aa0156901182143/notebooks/instance_segmentation/Ovarian_Reserve/model_weights/ovarian_reserve_pretrained.pth

For quick Colab testing, this notebook can reuse the small example dataset from the training workflow and run inference on its validation split.

If this workflow is useful for your research, please cite:

*3D Mapping of Intact Ovaries Reveals the Aging Dynamics of the Ovarian Reserve*  
Arturo D'Angelo, Daniel Franco-Barranco, Marco Musy, James Sharpe, Ignacio Arganda-Carreras, Elvan Boke  
bioRxiv 2025.11.07.686728; doi: https://doi.org/10.1101/2025.11.07.686728

## **Expected inputs and outputs**
___

### **Inputs**
This notebook expects:
- **Test Raw Images**: a folder containing the 3D TIFF volumes to segment.
- **[OPTIONAL] Test Label Images**: instance-label volumes matching the raw images, only if you want BiaPy to compute evaluation metrics.
- **Output Folder**: a directory where BiaPy will store predictions, post-processed results and logs.
- **Model Source**: an official ovarian reserve checkpoint, a custom `.pth` checkpoint, or a BMZ model ID.

### **Outputs**
If execution is successful, BiaPy creates result folders with:
- Predicted instance-segmentation channels.
- TIFF instance masks.
- Post-processed instance masks when post-processing is enabled.
- Optional matching statistics if ground truth labels are supplied.

### **Data structure**
A typical input directory tree is:
```
dataset/
  test/
    x/
      test-0001.tif
      test-0002.tif
      ...
    y/
      test-0001.tif
      test-0002.tif
      ...
```

Keep raw images and labels sorted in the same order when labels are provided.

This workflow is intended for 3D TIFF volumes.

## **Prepare the environment**
___

Establish connection with Google services. You must be logged in to Google to continue.
Since this is not Google's own code, Colab may warn about running external code. That is expected for this notebook.

## **Check for GPU access**
---

Verify in **Runtime -> Change runtime type**:
- **Runtime type**: Python 3
- **Hardware accelerator**: GPU

## **Install BiaPy**
---
This may take a few minutes depending on the current Colab environment.

In [ ]:
#@markdown ## Play to install BiaPy and its dependencies
!pip install biapy==3.6.8
!pip install torch==2.9.1 torchvision
!pip install timm==1.0.14 pytorch-msssim torchmetrics[image]==1.4.*

import os
import sys
import numpy as np
from tqdm.notebook import tqdm
from skimage.io import imread
import ipywidgets as widgets
from ipywidgets import Output
from biapy import BiaPy

uploaded_checkpoint_path = ''

## **Manage file(s) source**
---
The test data can be provided in three ways:
1. Upload files directly to Colab.
2. Mount Google Drive.
3. Download the example ovarian reserve dataset used by the training workflow and run inference on its validation split.

In [ ]:
#@markdown ## Play to upload local files (test raw images)
from google.colab import files
import os
input_dir = '/content/input/test/x'
os.makedirs(input_dir, exist_ok=True)
%cd {input_dir}
uploaded = files.upload()
%cd /content

In [ ]:
#@markdown ## Optional: upload local files (test labels)
from google.colab import files
import os
input_dir = '/content/input/test/y'
os.makedirs(input_dir, exist_ok=True)
%cd {input_dir}
uploaded = files.upload()
%cd /content

In [ ]:
#@markdown ## Play the cell to connect your Google Drive to Colab
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
#@markdown ## Play to download the example ovarian reserve dataset
import os
os.chdir('/content')
dataset_zip = 'oocyte_training.zip'
dataset_dir = '/content/oocyte_training'

print('Downloading example ovarian reserve dataset...')
if not os.path.exists(dataset_zip) and not os.path.exists(dataset_dir):
    !wget -q -O oocyte_training.zip 'https://upvehueus-my.sharepoint.com/:u:/g/personal/ignacio_arganda_ehu_eus/IQBlTg1-y8MlSqwgDpLZuPAgAU5oE0HOqc6vjDK7vVh_xBM?e=MMgzZf&download=1'
if os.path.exists(dataset_zip) and not os.path.exists(dataset_dir):
    !unzip -q oocyte_training.zip
print('Example dataset ready at: /content/oocyte_training')
print('Use /content/oocyte_training/val/raw as inference input and /content/oocyte_training/val/label as optional labels.')

## **Paths for input images and output files**
___

Depending on the option you chose, set the paths accordingly.

In [ ]:
#@markdown ##### Path to test raw images
test_data_path = '/content/oocyte_training/val/raw' #@param {type:"string"}
#@markdown ##### Select to load ground truth labels and compute metrics
load_gt = True #@param {type:"boolean"}
#@markdown ##### Path to optional test labels
test_data_gt_path = '/content/oocyte_training/val/label' #@param {type:"string"}
#@markdown ##### Output folder
output_path = '/content/output' #@param {type:"string"}

import os
from pathlib import Path

def count_image_files(directory):
    if not directory or not os.path.exists(directory):
        return 0
    image_extensions = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.npy', '.h5', '.hd5', '.zarr'}
    count = 0
    for root, dirs, files in os.walk(directory):
        for file in files:
            if Path(file).suffix.lower() in image_extensions:
                count += 1
    return count

num_test_images = count_image_files(test_data_path)
num_test_labels = count_image_files(test_data_gt_path) if load_gt else 0
print(f'Number of test raw images: {num_test_images}')
if load_gt:
    print(f'Number of test label images: {num_test_labels}')
    if num_test_images != num_test_labels:
        print('Warning: the number of raw images and labels does not match.')

In [ ]:
#@markdown ## Play to visualize the test data
%matplotlib inline
import numpy as np
from matplotlib import pyplot as plt
from skimage.io import imread
import os
from ipywidgets import IntSlider, Layout, VBox, Output

ids_input_all = sorted(next(os.walk(test_data_path))[2]) if os.path.exists(test_data_path) else []
ids_input = [f for f in ids_input_all if f.lower().endswith(('.tif', '.tiff'))]

if len(ids_input) == 0:
    print('No TIFF images found in the selected test path.')
else:
    input_img = imread(os.path.join(test_data_path, ids_input[0]))
    ids_gt = []
    gt_img = None
    if load_gt and os.path.exists(test_data_gt_path):
        ids_gt_all = sorted(next(os.walk(test_data_gt_path))[2])
        ids_gt = [f for f in ids_gt_all if f.lower().endswith(('.tif', '.tiff'))]
        if len(ids_gt) > 0:
            gt_img = imread(os.path.join(test_data_gt_path, ids_gt[0])).astype(np.uint16)

    vals = np.linspace(0, 1, 256)
    np.random.shuffle(vals)
    cmap = plt.cm.colors.ListedColormap(plt.cm.gist_rainbow(vals))
    cmap.colors[0] = [0.0, 0.0, 0.0, 1.0]

    sample_slider = IntSlider(value=1, min=1, max=len(ids_input), step=1, description='Image index:', continuous_update=False, layout=Layout(width='500px'))
    z_slider = IntSlider(value=1, min=1, max=len(input_img), step=1, description='Z value:', continuous_update=False, layout=Layout(width='500px'))
    sample_slider.style.description_width = 'initial'
    z_slider.style.description_width = 'initial'
    output = Output()
    state = {'input_img': input_img, 'gt_img': gt_img, 'index': 0}

    def update_sample(change):
        idx = change['new'] - 1
        state['index'] = idx
        state['input_img'] = imread(os.path.join(test_data_path, ids_input[idx]))
        if load_gt and len(ids_gt) > idx:
            state['gt_img'] = imread(os.path.join(test_data_gt_path, ids_gt[idx])).astype(np.uint16)
        else:
            state['gt_img'] = None
        z_slider.max = len(state['input_img'])
        z_slider.value = 1
        display_slice({'new': 1})

    def display_slice(change):
        with output:
            output.clear_output(wait=True)
            z = change['new'] - 1
            plt.figure(figsize=(12, 6))
            plt.subplot(1, 2, 1)
            plt.title(f'Raw image: {ids_input[state["index"]]} | z={z+1}')
            plt.imshow(state['input_img'][z], cmap='gray')
            plt.axis('off')
            if state['gt_img'] is not None:
                plt.subplot(1, 2, 2)
                plt.title('Label')
                plt.imshow(state['gt_img'][z], cmap=cmap, interpolation='nearest')
                plt.axis('off')
            plt.show()

    controls = VBox([sample_slider, z_slider])
    display(controls, output)
    sample_slider.observe(update_sample, names='value')
    z_slider.observe(display_slice, names='value')
    display_slice({'new': 1})

## **Select the model source**
---
The next cells let you pick between the official ovarian reserve checkpoint, a custom `.pth` checkpoint, or a BMZ model ID.

In [ ]:
#@markdown ### Optional: list BMZ models compatible with 3D instance segmentation in BiaPy
import pooch
from IPython.display import HTML, display
from biapy.models import check_bmz_model_compatibility, find_bmz_models

logger = pooch.get_logger()
logger.setLevel('WARNING')
workflow_specs = {'workflow_type': 'INSTANCE_SEG', 'ndim': '3D', 'nclasses': 'all'}
models = find_bmz_models()
compatible_models = []
restrictions = []
if len(models) > 0:
    print('Analyzing BMZ models...')
for model in tqdm(models, total=len(models)):
    try:
        preproc_info, error, error_message, model_imposed_vars = check_bmz_model_compatibility(model, workflow_specs=workflow_specs)
    except Exception:
        error = True
        model_imposed_vars = {}
    if not error:
        compatible_models.append(model)
        restrictions.append(model_imposed_vars)
html = "<table style='width:100%'>"
counter = 0
for i, model in enumerate(compatible_models):
    nickname = model.get('nickname', model.get('alias', 'Nickname not found'))
    extra = ''
    for key, val in restrictions[i].items():
        if key == 'PROBLEM.INSTANCE_SEG.DATA_CHANNELS':
            extra += f'<p>Data channels: {val}</p>'
        elif key == 'MODEL.N_CLASSES':
            extra += f'<p>Number of classes: {val}</p>'
    if counter == 0:
        html += '<tr>'
    html += f"<td style='width:33%'><p style='color:#1f77b4'>{model['raw']['manifest']['name']}</p><p>Nickname: {nickname}</p>{extra}</td>"
    counter += 1
    if counter == 3:
        html += '</tr>'
        counter = 0
html += '</table>'
if len(compatible_models) == 0:
    display(HTML('<h3>No BMZ models compatible with this 3D instance segmentation task were found at this time.</h3>'))
else:
    display(HTML('<h3>BMZ models compatible with 3D instance segmentation in BiaPy</h3>'))
    display(HTML(html))

In [ ]:
#@markdown ### Select the model source
model_source = 'Official pretrained model' #@param ['Official pretrained model', 'Custom .pth checkpoint', 'BioImage Model Zoo']
official_checkpoint_url = 'https://raw.githubusercontent.com/BiaPyX/BiaPy/8fd93d3167d061896edf940c7aa0156901182143/notebooks/instance_segmentation/Ovarian_Reserve/model_weights/ovarian_reserve_pretrained.pth' #@param {type:"string"}
custom_checkpoint_path = '' #@param {type:"string"}
bmz_model_id = '' #@param {type:"string"}
print(f'Selected model source: {model_source}')

In [ ]:
#@markdown ### Optional: upload a custom `.pth` checkpoint
from google.colab import files
import os
upload_dir = '/content/model_weights'
os.makedirs(upload_dir, exist_ok=True)
%cd {upload_dir}
uploaded = files.upload()
%cd /content
if len(uploaded) > 0:
    uploaded_checkpoint_path = os.path.join(upload_dir, list(uploaded.keys())[0])
    print(f'Uploaded checkpoint: {uploaded_checkpoint_path}')
else:
    print('No checkpoint uploaded in this step.')

In [ ]:
#@markdown ### Inference settings
patch_size_z = 40 #@param {type:"number"}
patch_size_xy = 128 #@param {type:"number"}
input_channels = 1 #@param {type:"number"}
voxel_size_z = 5.0 #@param {type:"number"}
voxel_size_y = 0.867 #@param {type:"number"}
voxel_size_x = 0.867 #@param {type:"number"}
padding_z = 10 #@param {type:"number"}
padding_y = 50 #@param {type:"number"}
padding_x = 50 #@param {type:"number"}
overlap_z = 0 #@param {type:"number"}
overlap_y = 0 #@param {type:"number"}
overlap_x = 0 #@param {type:"number"}
by_chunks = True #@param {type:"boolean"}
chunk_format = 'zarr' #@param ['zarr', 'h5']
input_axes_order = 'ZYX' #@param ['ZYX', 'TZCYX']
reduce_memory = True #@param {type:"boolean"}
in_memory = False #@param {type:"boolean"}
apply_post_processing = True #@param {type:"boolean"}
min_voxels = 150 #@param {type:"number"}
min_sphericity = 0.01 #@param {type:"number"}
max_voxels = 2000 #@param {type:"number"}

## **Run ovarian reserve inference**
---
This cell downloads the ovarian reserve inference template, adapts it to your paths and settings, resolves the model source, and runs BiaPy.

In [ ]:
#@markdown ## Play to configure and run ovarian reserve inference
import errno
import os
import yaml

job_name = 'ovarian_reserve_inference'
template_file = '/content/ovarian_reserve_inference_template.yaml'
inference_file = f'/content/{job_name}.yaml'
template_url = 'https://raw.githubusercontent.com/BiaPyX/BiaPy/master/templates/instance_segmentation/Ovarian_Reserve_paper/ovarian_reserve_inference.yaml'

for path in [template_file, inference_file]:
    if os.path.exists(path):
        os.remove(path)
!wget -q -O {template_file} {template_url}

if not os.path.exists(test_data_path):
    raise FileNotFoundError(errno.ENOENT, os.strerror(errno.ENOENT), test_data_path)
ids = sorted(next(os.walk(test_data_path))[2])
if len(ids) == 0:
    raise ValueError(f'No images found in dir {test_data_path}')
if load_gt:
    if not os.path.exists(test_data_gt_path):
        raise FileNotFoundError(errno.ENOENT, os.strerror(errno.ENOENT), test_data_gt_path)
    gt_ids = sorted(next(os.walk(test_data_gt_path))[2])
    if len(gt_ids) == 0:
        raise ValueError(f'No label images found in dir {test_data_gt_path}')

with open(template_file, 'r') as stream:
    biapy_config = yaml.safe_load(stream)

biapy_config['DATA']['TEST']['PATH'] = test_data_path
biapy_config['DATA']['TEST']['LOAD_GT'] = bool(load_gt)
biapy_config['DATA']['TEST']['IN_MEMORY'] = bool(in_memory)
biapy_config['DATA']['PATCH_SIZE'] = f'({patch_size_z},{patch_size_xy},{patch_size_xy},{input_channels})'
biapy_config['DATA']['TEST']['RESOLUTION'] = f'({voxel_size_z},{voxel_size_y},{voxel_size_x})'
biapy_config['DATA']['TEST']['PADDING'] = f'({padding_z},{padding_y},{padding_x})'
biapy_config['DATA']['TEST']['OVERLAP'] = f'({overlap_z},{overlap_y},{overlap_x})'
biapy_config['TRAIN']['ENABLE'] = False
biapy_config['TEST']['ENABLE'] = True
biapy_config['TEST']['REDUCE_MEMORY'] = bool(reduce_memory)
biapy_config['TEST']['BY_CHUNKS']['ENABLE'] = bool(by_chunks)
biapy_config['TEST']['BY_CHUNKS']['FORMAT'] = chunk_format
biapy_config['TEST']['BY_CHUNKS']['SAVE_OUT_TIF'] = True
biapy_config['TEST']['BY_CHUNKS']['INPUT_IMG_AXES_ORDER'] = input_axes_order
biapy_config['TEST']['BY_CHUNKS']['WORKFLOW_PROCESS']['ENABLE'] = bool(by_chunks)
biapy_config['TEST']['BY_CHUNKS']['WORKFLOW_PROCESS']['TYPE'] = 'entire_pred'
if load_gt:
    biapy_config['DATA']['TEST']['GT_PATH'] = test_data_gt_path
else:
    biapy_config['DATA']['TEST']['GT_PATH'] = ''
biapy_config['TEST']['POST_PROCESSING']['MEASURE_PROPERTIES']['ENABLE'] = bool(apply_post_processing)
biapy_config['TEST']['POST_PROCESSING']['MEASURE_PROPERTIES']['REMOVE_BY_PROPERTIES']['ENABLE'] = bool(apply_post_processing)
biapy_config['TEST']['POST_PROCESSING']['MEASURE_PROPERTIES']['REMOVE_BY_PROPERTIES']['VALUES'] = [[int(min_voxels)], [float(min_sphericity), int(max_voxels)]]

selected_checkpoint = ''
if model_source == 'Official pretrained model':
    selected_checkpoint = '/content/ovarian_reserve_pretrained.pth'
    if not os.path.exists(selected_checkpoint):
        !wget -q -O {selected_checkpoint} {official_checkpoint_url}
    biapy_config['MODEL']['SOURCE'] = 'biapy'
    biapy_config['MODEL']['LOAD_CHECKPOINT'] = True
    biapy_config['PATHS']['CHECKPOINT_FILE'] = selected_checkpoint
    biapy_config['MODEL'].setdefault('BMZ', {})
    biapy_config['MODEL']['BMZ']['SOURCE_MODEL_ID'] = ''
elif model_source == 'Custom .pth checkpoint':
    candidate_path = uploaded_checkpoint_path if uploaded_checkpoint_path != '' else custom_checkpoint_path.strip()
    if candidate_path == '':
        raise ValueError('Custom .pth checkpoint selected, but no checkpoint path was provided or uploaded.')
    if not os.path.exists(candidate_path):
        raise FileNotFoundError(errno.ENOENT, os.strerror(errno.ENOENT), candidate_path)
    selected_checkpoint = candidate_path
    biapy_config['MODEL']['SOURCE'] = 'biapy'
    biapy_config['MODEL']['LOAD_CHECKPOINT'] = True
    biapy_config['PATHS']['CHECKPOINT_FILE'] = selected_checkpoint
    biapy_config['MODEL'].setdefault('BMZ', {})
    biapy_config['MODEL']['BMZ']['SOURCE_MODEL_ID'] = ''
else:
    if bmz_model_id.strip() == '':
        raise ValueError('BioImage Model Zoo source selected, but bmz_model_id is empty.')
    biapy_config['MODEL']['SOURCE'] = 'bmz'
    biapy_config['MODEL']['LOAD_CHECKPOINT'] = False
    biapy_config['PATHS']['CHECKPOINT_FILE'] = ''
    biapy_config['MODEL'].setdefault('BMZ', {})
    biapy_config['MODEL']['BMZ']['SOURCE_MODEL_ID'] = str(bmz_model_id).strip()

with open(inference_file, 'w') as outfile:
    yaml.dump(biapy_config, outfile, default_flow_style=False)

print('Inference configuration finished.')
print(f'Configuration file: {inference_file}')
if selected_checkpoint != '':
    print(f'Checkpoint file: {selected_checkpoint}')
else:
    print(f'BMZ model ID: {bmz_model_id.strip()}')

biapy = BiaPy(inference_file, result_dir=output_path, name=job_name, run_id=1, gpu=0)
biapy.run_job()

In [ ]:
#@markdown ## Play to visualize inference results
%matplotlib inline
import os
import numpy as np
from matplotlib import pyplot as plt
from skimage.io import imread
from ipywidgets import IntSlider, Layout, VBox, Output

final_results = os.path.join(output_path, job_name, 'results', job_name + '_1')
candidate_dirs = [
    os.path.join(final_results, 'full_image_post_processing'),
    os.path.join(final_results, 'per_image_post_processing'),
    os.path.join(final_results, 'full_image_instances'),
    os.path.join(final_results, 'per_image_instances')
]
pred_dir = ''
for folder in candidate_dirs:
    if os.path.exists(folder):
        tif_files = [f for f in sorted(os.listdir(folder)) if f.lower().endswith(('.tif', '.tiff'))]
        if len(tif_files) > 0:
            pred_dir = folder
            break
if pred_dir == '':
    raise FileNotFoundError('No prediction TIFF files were found in the expected BiaPy result folders.')

ids_pred = [f for f in sorted(os.listdir(pred_dir)) if f.lower().endswith(('.tif', '.tiff'))]
raw_map = {os.path.basename(f): os.path.join(test_data_path, f) for f in sorted(os.listdir(test_data_path)) if f.lower().endswith(('.tif', '.tiff'))}
gt_map = {}
if load_gt and os.path.exists(test_data_gt_path):
    gt_map = {os.path.basename(f): os.path.join(test_data_gt_path, f) for f in sorted(os.listdir(test_data_gt_path)) if f.lower().endswith(('.tif', '.tiff'))}
first_name = ids_pred[0]
state = {
    'name': first_name,
    'raw': imread(raw_map[first_name]) if first_name in raw_map else None,
    'pred': imread(os.path.join(pred_dir, first_name)).astype(np.uint16),
    'gt': imread(gt_map[first_name]).astype(np.uint16) if first_name in gt_map else None
}
vals = np.linspace(0, 1, 256)
np.random.shuffle(vals)
cmap = plt.cm.colors.ListedColormap(plt.cm.gist_rainbow(vals))
cmap.colors[0] = [0.0, 0.0, 0.0, 1.0]
sample_slider = IntSlider(value=1, min=1, max=len(ids_pred), step=1, description='Prediction index:', continuous_update=False, layout=Layout(width='500px'))
z_slider = IntSlider(value=1, min=1, max=len(state['pred']), step=1, description='Z value:', continuous_update=False, layout=Layout(width='500px'))
sample_slider.style.description_width = 'initial'
z_slider.style.description_width = 'initial'
output = Output()

def update_prediction(change):
    idx = change['new'] - 1
    name = ids_pred[idx]
    state['name'] = name
    state['pred'] = imread(os.path.join(pred_dir, name)).astype(np.uint16)
    state['raw'] = imread(raw_map[name]) if name in raw_map else None
    state['gt'] = imread(gt_map[name]).astype(np.uint16) if name in gt_map else None
    z_slider.max = len(state['pred'])
    z_slider.value = 1
    display_prediction({'new': 1})

def display_prediction(change):
    with output:
        output.clear_output(wait=True)
        z = change['new'] - 1
        ncols = 4 if state['gt'] is not None else 2
        plt.figure(figsize=(5 * ncols, 5))
        col = 1
        if state['raw'] is not None:
            plt.subplot(1, ncols, col)
            plt.imshow(state['raw'][z], cmap='gray')
            plt.title(f'Raw | {state["name"]} | z={z+1}')
            plt.axis('off')
            col += 1
        plt.subplot(1, ncols, col)
        plt.imshow(state['pred'][z], cmap=cmap, interpolation='nearest')
        plt.title('Prediction')
        plt.axis('off')
        col += 1
        if state['gt'] is not None:
            plt.subplot(1, ncols, col)
            plt.imshow(state['gt'][z], cmap=cmap, interpolation='nearest')
            plt.title('Ground truth')
            plt.axis('off')
            col += 1
            plt.subplot(1, ncols, col)
            plt.imshow(state['gt'][z], cmap='Greens')
            plt.imshow(state['pred'][z], alpha=0.5, cmap='Purples')
            plt.title('Overlay')
            plt.axis('off')
        plt.show()

display(VBox([sample_slider, z_slider]), output)
sample_slider.observe(update_prediction, names='value')
z_slider.observe(display_prediction, names='value')
display_prediction({'new': 1})

In [ ]:
#@markdown ## Play to download the inference result folder as a ZIP file
from google.colab import files
import os
final_results = os.path.join(output_path, job_name, 'results', job_name + '_1')
download_zip = '/content/ovarian_reserve_inference_results.zip'
if not os.path.exists(final_results):
    raise FileNotFoundError(f'BiaPy results folder not found: {final_results}')
if os.path.exists(download_zip):
    os.remove(download_zip)
!zip -q -r {download_zip} {final_results}
files.download(download_zip)

In [ ]:
#@markdown ## Play to download the generated inference YAML file
from google.colab import files
files.download(inference_file)